In [1]:
from sts.dio.equity import TickerDatabase as Ticker
import plotly.io as pio
import pandas as pd
import numpy as np
from sts.quant.consolidate import get_con_score_mm_multi_horizon
from sts.quant.breakout import get_breakout_mm_multi_horizon
from sts.quant.candle import Candle
import os
import plotly.graph_objs as go
from sts.dio.symbols import sp500_symbol_table
from IPython.display import clear_output

pd.options.plotting.backend = "plotly"
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook")

In [ ]:
resample_rule = "1D"
rule_period_map = {"1D": "2y", "1W": "10y", "1M": "20y"}
period = rule_period_map[resample_rule]
consolidate_score_all_df = []
future = ["^SPX", "^NDX", "^RUT", "GC=F", "HG=F", "SI=F", "CL=F"]
etf = [
    "SPY",
    "QQQ",
    "IWM",
    "GLD",
    "SLV",
    "CPER",
    "USO",
    "TLT",
    "KWEB",
    "PPA",
    "IBIT",
    "XLF",
    "XLI",
    "XLB",
    "XLC",
    "XLY",
    "XLP",
    "XLV",
    "XBI",
    "XLK",
    "XLU",
    "XME",
]
fx = ["JPY=X", "GBPUSD=X", "AUDUSD=X", "EURUSD=X", "CADUSD=X"]
for sym in etf:  # future + etf + fx:
    try:
        ticker = Ticker(sym)
        df = Candle(ticker.history(period=period))
        con_score_dict = get_con_score_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.2)
        breakout_score_dict = get_breakout_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.1)
        score = {}
        score["Sym"] = sym
        for key in con_score_dict.keys():
            ts = con_score_dict[key]
            ts = ts.rolling(window=3).mean()
            score[f"Con{key}"] = ts.iloc[-1]
        for key in breakout_score_dict.keys():
            ts = breakout_score_dict[key]
            ts = ts.rolling(window=3).mean()
            score[f"Break{key}"] = ts.iloc[-1]
        consolidate_score_all_df.append(score)
        clear_output()
    except Exception as e:
        print(e)
consolidate_score_all_df = pd.DataFrame(consolidate_score_all_df)

Too Many Requests. Rate limited. Try after a while.


In [25]:
consolidate_score_all_df = consolidate_score_all_df.rename(
    columns={
        "ConDailyScore": "CDScore",
        "ConWeeklyScore": "CWScore",
        "ConMonthlyScore": "CMScore",
        "BreakDailyScore": "BDScore",
        "BreakWeeklyScore": "BWScore",
        "BreakMonthlyScore": "BMScore",
    }
)

In [26]:
con_score_dict["MonthlyScore"].plot()

* Weekly consolidating

In [11]:
sub_df = consolidate_score_all_df[(consolidate_score_all_df["CWScore"] > 0.8)]
sub_df.sort_values("BDScore", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore
7,TLT,1.0,1.0,1.0,0.0,0.0,0.0
8,KWEB,1.0,1.0,0.0,0.0,0.0,0.0
16,TLT,1.0,1.0,1.0,0.0,0.0,0.0
17,KWEB,1.0,1.0,0.0,0.0,0.0,0.0
24,XLP,1.0,1.0,0.0,0.0,0.0,1.0


* Weekly consolidating and daily breakout

In [12]:
sub_df = consolidate_score_all_df[
    (consolidate_score_all_df["CWScore"] > 0.8) & (consolidate_score_all_df["BDScore"] > 0.5)
]
sub_df.sort_values("BDScore", ascending=False).round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore


* monthly trending and weekly consolidating and daily breakout

In [6]:
consolidate_score_all_df[
    (consolidate_score_all_df["BMScore"].abs() > 0.6)
    & (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["BDScore"].abs() > 0.5)
].round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore


* monthly consolidating and weekly breakout

In [7]:
consolidate_score_all_df[
    (consolidate_score_all_df["CMScore"] > 0.6)
    & (consolidate_score_all_df["CWScore"] > 0.6)
    & (consolidate_score_all_df["BDScore"].abs() > 0.5)
].round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore


* Monthly Trending

In [14]:
consolidate_score_all_df[(consolidate_score_all_df["BMScore"].abs() > 0.8)].round(2)

,Sym,CDScore,CWScore,CMScore,BDScore,BWScore,BMScore
0,^SPX,0.33,0.00,0.0,0.00,-0.33,1.0
3,GC=F,0.33,0.00,0.0,0.67,1.00,1.0
5,SI=F,1.00,0.00,0.0,0.00,0.00,1.0
9,SPY,0.67,0.00,0.0,0.00,-0.33,1.0
10,QQQ,0.33,0.00,0.0,0.00,-0.33,1.0
12,GLD,0.33,0.00,0.0,0.67,1.00,1.0
13,SLV,1.00,0.67,0.0,0.00,0.00,1.0
18,PPA,0.00,0.00,0.0,0.33,0.00,1.0
19,XLF,0.67,0.00,0.0,0.00,0.00,1.0
20,XLI,0.33,0.00,0.0,0.00,0.00,1.0
